In [2]:
#Fake Review Detector

In [4]:
import pandas as pd
import re
import string
import nltk

In [6]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [10]:
from sklearn.model_selection import train_test_split

In [12]:
from sklearn.linear_model import LogisticRegression

In [14]:
from sklearn.metrics import classification_report, accuracy_score

In [16]:
nltk.download('punkt')#performs basic level tokenization and can't handle complex data
nltk.download('stopwords')#downloads common stopwords
nltk.download('wordnet')#downloads wordnet for lemmatization
nltk.download('punkt_tab')#performs high level tokenization and can handle complex data

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/bhanuni.sunkaraicloud.com/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/bhanuni.sunkaraicloud.com/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/bhanuni.sunkaraicloud.com/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/bhanuni.sunkaraicloud.com/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [18]:
df = pd.read_csv('/Users/bhanuni.sunkaraicloud.com/Desktop/deceptive-opinion.csv')

In [20]:
print("Original Data Sample:")
print(df[['text', 'polarity']].head(20))

Original Data Sample:
                                                 text  polarity
0   We stayed for a one night getaway with family ...  positive
1   Triple A rate with upgrade to view room was le...  positive
2   This comes a little late as I'm finally catchi...  positive
3   The Omni Chicago really delivers on all fronts...  positive
4   I asked for a high floor away from the elevato...  positive
5   I stayed at the Omni for one night following a...  positive
6   We stayed in the Conrad for 4 nights just befo...  positive
7   Just got back from 2 days up in Chicago shoppi...  positive
8   We arrived at the Omni on 2nd September for a ...  positive
9   On our visit to Chicago, we chose the Hyatt du...  positive
10  I stayed at the Fairmont Chicago for one night...  positive
11  Ok, so first trip to chicago and I was a litll...  positive
12  We arrived at 10:30 am on a Friday, and they h...  positive
13  My wife and I came to spend the weekend in dow...  positive
14  I got a Sunday

In [22]:
df['label'] = df['polarity'].map({'positive': 1, 'negative': 0})

In [24]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [26]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r'\d+', '', text)
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return " ".join(tokens)

In [28]:
df['clean_text'] = df['text'].apply(preprocess_text)

In [30]:
print("\nCleaned Data with Labels:")
print(df[['clean_text', 'label']].head(20))


Cleaned Data with Labels:
                                           clean_text  label
0   stayed one night getaway family thursday tripl...      1
1   triple rate upgrade view room less also includ...      1
2   come little late im finally catching review pa...      1
3   omni chicago really delivers front spaciousnes...      1
4   asked high floor away elevator got room pleasa...      1
5   stayed omni one night following business meeti...      1
6   stayed conrad night thanksgiving corner room o...      1
7   got back day chicago shopping girlfriend first...      1
8   arrived omni nd september day stay took ill le...      1
9   visit chicago chose hyatt due location downtow...      1
10  stayed fairmont chicago one night im frequent ...      1
11  ok first trip chicago litlle worried hotel loc...      1
12  arrived friday room ready u mind check pm got ...      1
13  wife came spend weekend downtown chicago shopp...      1
14  got sunday night stay pricelinecom would hard ...     

In [32]:
#Feature Extraction
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['clean_text'])
y = df['label']

In [34]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [36]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [38]:
print("\nResults for Logistic Regression:")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))


Results for Logistic Regression:
Accuracy: 0.94375
Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.95      0.94       149
           1       0.96      0.94      0.95       171

    accuracy                           0.94       320
   macro avg       0.94      0.94      0.94       320
weighted avg       0.94      0.94      0.94       320



In [44]:
import pickle
import os
os.makedirs('model', exist_ok=True)

with open('model/logistic_model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('model/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
